# Day 4: colorectal histopathology with external validation
Runtime > Change runtime type > **T4 GPU**, then run the cells in order. About 15 GB of downloads and roughly 1 hour in total (estimates). Protocol and the pre-registered decision are in the README.

In [ ]:
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
!pip -q install huggingface_hub pyarrow
!if [ -d /content/MedTech-Suite ]; then cd /content/MedTech-Suite && git fetch -q origin && git checkout origin/main -- day-4-crc-histopathology/src; else git clone -q https://github.com/aryan9-6-5/MedTech-Suite.git /content/MedTech-Suite; fi
%cd /content/MedTech-Suite/day-4-crc-histopathology/src

In [ ]:
# download the two image sets (about 14 GB from the Hugging Face mirror), assert the class counts match Zenodo's,
# then cache a balanced subsample as uint8 arrays
!python data.py
!ls -la ../data/cache

In [ ]:
# four runs: arms A and B, seeds 42 and 7 (about 10 min each)
!ARM=A SEED=42 python train.py
!ARM=B SEED=42 python train.py
!ARM=A SEED=7 python train.py
!ARM=B SEED=7 python train.py

In [ ]:
!python evaluate.py

In [ ]:
# how close are internal vs external test patches to the training set, and Grad-CAM examples
!python leak.py
!python gradcam.py

In [ ]:
%cd /content/MedTech-Suite/day-4-crc-histopathology
import json
from IPython.display import Image, display
print(open('results/summary.json').read()[:3500])
print(open('results/leakage.json').read())
for f in ('confusion_external','gradcam_external'): display(Image(f'results/{f}.png'))